# Session 37 Solutions: Decision Trees and Random Forests

Data Science with Python, Level 3, Stage 6: Applied ML

Try each exercise yourself before reading its solution. There is usually more than one correct answer; compare your approach with this one.

## Setup

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, plot_tree, export_text
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import roc_auc_score

DATA_URL = "https://raw.githubusercontent.com/mohammadyahoo/ds-python-course/main/data/"
churn = pd.read_csv(DATA_URL + "telecom_churn.csv")
churn["left"] = (churn["churn"] == "Yes").astype(int)
churn["monthly_contract"] = (churn["contract"] == "Month-to-month").astype(int)
churn["fiber"] = (churn["internet"] == "Fiber").astype(int)
churn["echeck"] = (churn["payment_method"] == "Electronic check").astype(int)
features = ["tenure_months", "monthly_charges", "support_calls", "senior",
            "monthly_contract", "fiber", "echeck"]
X = churn[features].astype(float)
y = churn["left"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)

## In-Class Exercises

### In-Class Exercise 1: Read the Tree

In [2]:
t = DecisionTreeClassifier(max_depth=3, random_state=0).fit(X_train, y_train)
print(export_text(t, feature_names=features, show_weights=True))

|--- monthly_contract <= 0.50
|   |--- tenure_months <= 40.50
|   |   |--- monthly_charges <= 83.17
|   |   |   |--- weights: [231.00, 37.00] class: 0
|   |   |--- monthly_charges >  83.17
|   |   |   |--- weights: [67.00, 26.00] class: 0
|   |--- tenure_months >  40.50
|   |   |--- monthly_charges <= 84.55
|   |   |   |--- weights: [218.00, 7.00] class: 0
|   |   |--- monthly_charges >  84.55
|   |   |   |--- weights: [69.00, 12.00] class: 0
|--- monthly_contract >  0.50
|   |--- monthly_charges <= 75.30
|   |   |--- tenure_months <= 21.50
|   |   |   |--- weights: [135.00, 104.00] class: 0
|   |   |--- tenure_months >  21.50
|   |   |   |--- weights: [174.00, 59.00] class: 0
|   |--- monthly_charges >  75.30
|   |   |--- tenure_months <= 27.50
|   |   |   |--- weights: [89.00, 160.00] class: 1
|   |   |--- tenure_months >  27.50
|   |   |   |--- weights: [62.00, 50.00] class: 0



### In-Class Exercise 2: Depth Search

In [3]:
for d in range(1, 11):
    s = cross_val_score(DecisionTreeClassifier(max_depth=d, random_state=0), X_train, y_train, cv=5, scoring="roc_auc")
    print(d, round(s.mean(), 3))

1 0.69
2 0.738
3 0.746
4 0.73
5 0.714
6 0.703
7 0.69
8 0.669
9 0.671
10 0.65


### In-Class Exercise 3: Forest for Prices

In [4]:
flats = pd.read_csv(DATA_URL + "apartments.csv")
cols = ["area_m2", "rooms", "floor", "building_age", "distance_center_km", "elevator", "parking"]
for m in [DecisionTreeRegressor(max_depth=5, random_state=0), RandomForestRegressor(n_estimators=300, random_state=0, n_jobs=-1)]:
    print(type(m).__name__, round(cross_val_score(m, flats[cols].astype(float), flats["price_k"], cv=5, scoring="r2").mean(), 3))

DecisionTreeRegressor 0.587
RandomForestRegressor 0.673


### In-Class Exercise 4: Leaf Size

In [5]:
for leaf in [1, 5, 10, 30]:
    f = RandomForestClassifier(n_estimators=200, min_samples_leaf=leaf, random_state=0, n_jobs=-1).fit(X_train, y_train)
    print(leaf, round(f.score(X_train, y_train), 3), round(roc_auc_score(y_test, f.predict_proba(X_test)[:, 1]), 3))

1 1.0 0.761
5 0.827 0.804
10 0.785 0.808
30 0.753 0.81


## Part A: Basic

### Exercise 1: Your First Tree

In [6]:
exams = pd.read_csv(DATA_URL + "exam_scores.csv")
cols = ["study_hours_week", "sleep_hours", "attendance_pct"]
t = DecisionTreeClassifier(max_depth=2, random_state=0).fit(exams[cols], exams["passed"])
print(export_text(t, feature_names=cols))

|--- study_hours_week <= 10.05
|   |--- study_hours_week <= 4.65
|   |   |--- class: False
|   |--- study_hours_week >  4.65
|   |   |--- class: False
|--- study_hours_week >  10.05
|   |--- study_hours_week <= 10.90
|   |   |--- class: True
|   |--- study_hours_week >  10.90
|   |   |--- class: True



### Exercise 2: Predict One Customer

In [7]:
tree = DecisionTreeClassifier(max_depth=2, random_state=0).fit(X_train, y_train)
c = pd.DataFrame([{"tenure_months": 5, "monthly_charges": 90, "support_calls": 2, "senior": 0, "monthly_contract": 1, "fiber": 1, "echeck": 1}])[features]
print(tree.predict(c), tree.predict_proba(c).round(3))

[1] [[0.418 0.582]]


## Part B: Intermediate

### Exercise 3: Fraud Forest

Fraud happens at night and far from home: a pattern of thresholds that trees capture well, so the forest beats the linear model here.

In [8]:
tx = pd.read_csv(DATA_URL + "card_transactions.csv")
cols = ["amount", "hour", "distance_from_home_km", "online", "new_merchant"]
Xf_train, Xf_test, yf_train, yf_test = train_test_split(tx[cols].astype(float), tx["is_fraud"], test_size=0.25, random_state=1, stratify=tx["is_fraud"])
f = RandomForestClassifier(n_estimators=300, min_samples_leaf=5, random_state=0, n_jobs=-1).fit(Xf_train, yf_train)
print(round(roc_auc_score(yf_test, f.predict_proba(Xf_test)[:, 1]), 3))
print(pd.Series(f.feature_importances_, index=cols).sort_values(ascending=False).round(3))
lr = make_pipeline(StandardScaler(), LogisticRegression()).fit(Xf_train, yf_train)
print("logistic", round(roc_auc_score(yf_test, lr.predict_proba(Xf_test)[:, 1]), 3))

1.0
distance_from_home_km    0.694
amount                   0.248
hour                     0.036
new_merchant             0.016
online                   0.007
dtype: float64
logistic 1.0


### Exercise 4: Bike Rentals Forest

In [9]:
from sklearn.linear_model import LinearRegression
bikes = pd.read_csv(DATA_URL + "bike_rentals.csv")
cols = ["temp_c", "humidity", "wind_kmh", "working_day", "holiday"]
Xb = bikes[cols].astype(float)
for m in [LinearRegression(), RandomForestRegressor(n_estimators=300, min_samples_leaf=5, random_state=0, n_jobs=-1)]:
    print(type(m).__name__, round(cross_val_score(m, Xb, bikes["rentals"], cv=5, scoring="r2").mean(), 3))
# Rentals rise with temperature and then fall on very hot days: a curve a straight line cannot follow.

LinearRegression 0.455
RandomForestRegressor 0.476


## Part C: Challenge

### Exercise 5: A Forest by Hand

In [10]:
probs = []
for i in range(25):
    rows = X_train.sample(frac=1, replace=True, random_state=i)
    t = DecisionTreeClassifier(max_depth=6, max_features="sqrt", random_state=i).fit(rows, y_train.loc[rows.index])
    probs.append(t.predict_proba(X_test)[:, 1])
print("my forest", round(roc_auc_score(y_test, np.mean(probs, axis=0)), 3))
one = DecisionTreeClassifier(max_depth=6, random_state=0).fit(X_train, y_train)
print("one tree", round(roc_auc_score(y_test, one.predict_proba(X_test)[:, 1]), 3))

my forest 0.806
one tree 0.762


## Part D: Cumulative Project: Customer Churn

### Exercise 6: Customer Churn: Step 2, Trying a Forest

In [11]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
churn = pd.read_csv(DATA_URL + "telecom_churn.csv")
y = (churn["churn"] == "Yes").astype(int)
X = churn.drop(columns=["customer_id", "churn"])
X_dev, X_final, y_dev, y_final = train_test_split(X, y, test_size=0.2, stratify=y, random_state=0)
num_cols = ["tenure_months", "monthly_charges", "total_charges", "support_calls"]
cat_cols = ["gender", "contract", "internet", "payment_method"]
bool_cols = ["senior", "paperless_billing"]
def preprocess():
    return ColumnTransformer([
        ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), num_cols),
        ("cat", make_pipeline(SimpleImputer(strategy="constant", fill_value="None"), OneHotEncoder(handle_unknown="ignore")), cat_cols),
        ("bool", "passthrough", bool_cols),
    ])
logreg = make_pipeline(preprocess(), LogisticRegression(max_iter=1000))
forest = make_pipeline(preprocess(), RandomForestClassifier(n_estimators=300, min_samples_leaf=10, random_state=0, n_jobs=-1))
for name, m in [("logistic", logreg), ("forest", forest)]:
    print(name, cross_val_score(m, X_dev, y_dev, cv=5, scoring="roc_auc").mean().round(3))
forest.fit(X_dev, y_dev)
imp = pd.Series(forest[-1].feature_importances_, index=forest[:-1].get_feature_names_out())
imp.sort_values(ascending=False).head(10).round(3)

logistic 0.786
forest 0.774


num__tenure_months                      0.201
num__monthly_charges                    0.162
cat__contract_Month-to-month            0.159
num__total_charges                      0.126
num__support_calls                      0.064
cat__contract_Two year                  0.058
cat__internet_Fiber                     0.041
cat__payment_method_Electronic check    0.041
cat__contract_One year                  0.034
cat__internet_None                      0.016
dtype: float64

## Part E: Quiz Answers

| Question | Answer | Explanation |
| --- | --- | --- |
| 1 | B | Each leaf holds the prediction. |
| 2 | B | It can memorise every training row. |
| 3 | B | Bootstrap rows and random feature subsets. |
| 4 | B | Splits depend on thresholds, not scale. |
| 5 | B | Importance is about use, not cause or direction. |